### Harry potter rag 

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY") # This is the API key for the Gemini API, which is used to access the language model.

csv_path = "data/harry_potter_cleaned.csv" 

df = pd.read_csv(csv_path)
df.head() 

,name,gender,job,house,patronus,species,blood_status,loyalty,skills,birth,...,titles,category,hand,light,difficulty,inventors,ingredients,side_effects,time,title
0,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gryffindor,Phoenix,Human,Half-blood,Dumbledore's Army | Order of the Phoenix | Hog...,Considered by many to be one of the most power...,Late August 1881,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Quirinus Quirrell,Male,Defence Against the Dark Arts(1991-1992),Ravenclaw,Non-corporeal,Human,Half-blood,Lord Voldemort,Learned in the theory of Defensive Magic| less...,"26 September,1970 or earlier",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fred Weasley,Male,Student,Gryffindor,Unknown,Human,Pure-blood,Dumbledore's Army | Order of the Phoenix | Hog...,Beater,"1 April, 1978",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Nymphadora Tonks,Female,Auror,Hufflepuff,"Jack rabbit, Wolf",Human,Half-blood,Ministry of Magic | Order of the Phoenix,"Talented Auror, Metamorphmagus",1 September 1972- 31 August 1973,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Lavender Brown,Female,Student,Gryffindor,Unknown,Human,Pure-blood,Dumbledore's Army |Hogwarts School of Witchcra...,Divination,1 September 1979- 31 August 1980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Load the CSV file using the CSVLoader from langchain_community
from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path=csv_path)
documents = loader.load()
print(f"Number of documents: {len(documents)}")
print(f"First document: {documents[0]}")

/Users/opheliathomasson/Documents/TUC_datamanager/datakvalité/kunskapskontroll/data_kvalit-_kunskapskontroll/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Number of documents: 5766
First document: page_content='name: Albus Percival Wulfric Brian Dumbledore
gender: Male
job: Headmaster
house: Gryffindor
patronus: Phoenix
species: Human
blood_status: Half-blood
loyalty: Dumbledore's Army | Order of the Phoenix | Hogwarts School of Witchcraft and Wizardry
skills: Considered by many to be one of the most powerful wizards of his time
birth: Late August 1881
death: 30 June, 1997
source: characters
incantation: 
type: 
effect: 
characteristics: 
alias_names: 
titles: 
category: 
hand: 
light: 
difficulty: 
inventors: 
ingredients: 
side_effects: 
time: 
title: ' metadata={'source': 'data/harry_potter_cleaned.csv', 'row': 0}


In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader

book_documents = []

for file_path in Path("data").glob("*.txt"): # Loop through all text files in the "data" directory
    
    loader = TextLoader(file_path, encoding="utf-8")
    docs = loader.load()

    for doc in docs:
        doc.metadata["source"] = file_path.name

    book_documents.extend(docs)

print(len(book_documents))

7


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

book_chunks = text_splitter.split_documents(book_documents)

print(len(book_chunks))

10965


In [5]:
all_documents = documents + book_chunks # Combine the documents from the CSV file and the text files into a single list

In [6]:
# use the HuggingFaceEmbeddings class from langchain_community to create an embeddings model instead of the GoogleGenerativeAIEmbeddings 
# class from langchain_google_genai, since the latter is not working because the limit for the free API key has been reached. The HuggingFaceEmbeddings class 
# allows us to use a variety of pre-trained models from Hugging Face, including the sentence-transformers models, which are designed for generating embeddings for 
# sentences and paragraphs.

from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/p5/jkbyc74n1jj4grxpk0w1ltf80000gn/T/ipykernel_95384/1062252735.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6605.91it/s]


In [7]:
# Create a Chroma vector store and add the documents to it
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="harry_potter_hf",
    embedding_function=embeddings,
    persist_directory="./chroma_harry_potter_test_db",
)

In [8]:
# Entering documents into the vector data base
batch_size = 500

for i in range(0, len(all_documents), batch_size):
    batch = all_documents[i:i + batch_size]

    vector_store.add_documents(documents=batch)

    print(f"Added documents {i} to {i + len(batch)}")

Added documents 0 to 500
Added documents 500 to 1000
Added documents 1000 to 1500
Added documents 1500 to 2000
Added documents 2000 to 2500
Added documents 2500 to 3000
Added documents 3000 to 3500
Added documents 3500 to 4000
Added documents 4000 to 4500
Added documents 4500 to 5000
Added documents 5000 to 5500
Added documents 5500 to 6000
Added documents 6000 to 6500
Added documents 6500 to 7000
Added documents 7000 to 7500
Added documents 7500 to 8000
Added documents 8000 to 8500
Added documents 8500 to 9000
Added documents 9000 to 9500
Added documents 9500 to 10000
Added documents 10000 to 10500
Added documents 10500 to 11000
Added documents 11000 to 11500
Added documents 11500 to 12000
Added documents 12000 to 12500
Added documents 12500 to 13000
Added documents 13000 to 13500
Added documents 13500 to 14000
Added documents 14000 to 14500
Added documents 14500 to 15000
Added documents 15000 to 15500
Added documents 15500 to 16000
Added documents 16000 to 16500
Added documents 16500

In [9]:
# Checking the similarity search with a sample question.
question = "What house does Harry Potter belong to?"

results = vector_store.similarity_search(question, k=3)

for result in results:
    print(result.page_content)
    print("---")

Page | 75 Harry Potter and the Half Blood Prince - J.K. Rowling 




Muggle house — the owners of this place are on 
holiday in the Canary Islands — it’s been very 
pleasant, I’ll be sorry to leave. It’s quite easy once you 
know how, one simple Freezing Charm on these 
absurd burglar alarms they use instead of 
Sneakoscopes and make sure the neighbors don’t 
spot you bringing in the piano.” 

“Ingenious,” said Dumbledore. “But it sounds a rather 
tiring existence for a broken-down old buffer in 
search of a quiet life. Now, if you were to return to 
Hogwarts — ”
---
Harry understood completely. He knew how he would 
feel if forced, when he was grown up and thought he 
was free of the place forever, to return and live at 
number four, Privet Drive. 

“It’s ideal for headquarters, of course,” Sirius said. 

“My father put every security measure known to 
Wizard-kind on it when he lived here. It’s Unplottable, 
so Muggles could never come and call — as if they’d 
have wanted to — and now

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    google_api_key=api_key
)

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [12]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

template = """
You are an old and wise wizard from the magical world.
Answer the user's question in a mysterious but helpful tone, and keep the answer concise.

Use only the information in the retrieved data.
If the answer is not found in the retrieved data, say that even your ancient magic cannot find the answer.

Retrieved data:
{retrieved_data}

Question:
{question}
"""


In [13]:
prompt = PromptTemplate.from_template(template)
parser = StrOutputParser()

In [14]:
chain = (
    {
        "retrieved_data": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | parser
)

In [21]:
question = input("Ask a question about the Harry Potter dataset: ")


In [22]:
answer = chain.invoke(question)

In [23]:
print(answer)

Seek the brew known as Felix Felicis, seeker of fortune. A mere bottle provides twelve hours of luck in all you attempt, turning the ordinary into the extraordinary. Yet heed my warning: such magic is forbidden in competitions, examinations, and elections alike.
